# W06 · Pink-PEPS & the 1/f story / Pink-PEPS 與 1/f

**English.** Natural images have power spectra that fall off roughly as
`1/f^alpha`. Pink-PEPS exploits this by allocating latent capacity **inversely
to frequency**, so it matches Grid-PEPS quality with fewer parameters (the
paper's ~-25% result). We first verify the 1/f slope (Fig. 3), then train Pink.

**繁體中文.** 自然影像的功率譜大致以 `1/f^alpha` 衰減。Pink-PEPS 依此把 latent
容量**與頻率成反比**分配,以更少參數達到 Grid-PEPS 品質(論文約 -25%)。先驗證
1/f 斜率(Fig.3),再訓練 Pink。

In [1]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
import torch, matplotlib.pyplot as plt
from peps.train import auto_device
device = auto_device(); print('device', device)

device cuda


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


## 1. Verify 1/f PSD on Kodak (reproduce Fig. 3) / 驗證 1/f(重現 Fig.3)

In [2]:
from apps.image.data import load_image, find_kodak
from peps.spectral import radial_psd, fit_one_over_f
import numpy as np
img = load_image(find_kodak(1), max_size=512)
freqs, psd = radial_psd(img, nbins=80)
alpha = fit_one_over_f(freqs, psd)
print(f'estimated 1/f slope alpha = {alpha:.2f}')
plt.loglog(freqs, psd, '.'); plt.xlabel('radial freq'); plt.ylabel('power')
plt.title(f'Radial PSD (slope ~ {alpha:.2f})'); plt.grid(True, which='both', alpha=0.3); plt.show()

estimated 1/f slope alpha = 2.45


## 2. Train Pink-PEPS vs Grid-PEPS / 訓練 Pink 對比 Grid

In [3]:
from apps.image.data import image_to_coords_targets
from apps.image.build import build_grid_peps
from peps.train import fit, TrainConfig, render_full
from peps.metrics import psnr
coords, targets, (H, W) = image_to_coords_targets(img)
def run(agg):
    model, pc = build_grid_peps(resolution=128, feature_dim=8, num_frequencies=6, aggregator=agg)
    fit(model, coords, targets, TrainConfig(steps=2500, batch_size=32768, lr=1e-2, device=device))
    pred = render_full(model, coords, device=device).reshape(H, W, 3).clamp(0, 1)
    return pc, psnr(pred, img)
c_pc, c_ps = run('concat')
p_pc, p_ps = run('pink')
print(f'concat: params={c_pc}  PSNR={c_ps:.2f}')
print(f'pink  : params={p_pc}  PSNR={p_ps:.2f}  ({100*(1-p_pc/c_pc):.0f}% fewer params)')

concat: params=142147  PSNR=35.44
pink  : params=138563  PSNR=35.39  (3% fewer params)


## 3. Takeaway / 小結
Pink matches concat quality with fewer parameters — capacity follows the
signal's spectrum. Pink 以更少參數達到 concat 品質,容量跟著訊號頻譜走。